# Lesson 2.3 — CloudTrail, EventBridge & SNS Notifications

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Configure AWS CloudTrail to capture API activity and management events
- Query CloudTrail events using the CLI and boto3
- Build EventBridge rules that route events to targets (Lambda, SNS, SQS)
- Create end-to-end event-driven notification pipelines
- Troubleshoot EventBridge rule issues

**Exam relevance:** SOA-C03 Domain 1, Skills 1.1.3–1.1.5; Domain 4, Skill 4.1.2

_Reference: [CloudTrail Docs](https://docs.aws.amazon.com/cloudtrail/latest/userguide/) | [EventBridge Docs](https://docs.aws.amazon.com/eventbridge/latest/userguide/)_

## 1. AWS CloudTrail

**CloudTrail** records every API call made in your AWS account — who did what, when, from where. Think of it as the security camera footage for your entire AWS account.

### Event Types
| Type | What it captures | Cost |
|------|-----------------|------|
| **Management Events** | Control-plane actions (creating EC2, modifying IAM) | Free for first copy per region |
| **Data Events** | Data-plane actions (S3 GetObject, Lambda Invoke) | $0.10 per 100K events |
| **Insights Events** | Unusual API activity detection | $0.35 per 100K |

### Trail Configuration
- **Single-region trail**: Only captures events in one region
- **Multi-region trail**: Captures all regions (recommended)
- **Organization trail**: Captures events across all accounts in AWS Organizations
- Events are delivered to **S3** (within 15 minutes) and optionally **CloudWatch Logs** (near real-time)

### Key Security Uses
- Who deleted that S3 bucket? (`LookupEvents` on `DeleteBucket`)
- Who changed that security group? (`AuthorizeSecurityGroupIngress`)
- Root account used? (`userType=Root`)

---

## 2. Amazon EventBridge

**EventBridge** (formerly CloudWatch Events) is a serverless event bus that routes events between AWS services, your applications, and third-party SaaS tools.

### Architecture
```
Event Sources                    EventBridge Bus         Targets
─────────────                    ───────────────         ───────
AWS Services (EC2, S3, RDS)  →   default bus        →   Lambda
Custom Applications          →   custom bus         →   SNS/SQS
SaaS Partners (Zendesk)      →   partner bus        →   ECS Task
                                                    →   Step Functions
                                                    →   API Destination (webhook)
```

### Event Pattern Matching
EventBridge rules filter events using **JSON pattern matching**:
```json
{
  "source": ["aws.ec2"],
  "detail-type": ["EC2 Instance State-change Notification"],
  "detail": {
    "state": ["terminated"]
  }
}
```

In [ ]:
import boto3
import json
from datetime import datetime, timedelta, timezone

cloudtrail = boto3.client('cloudtrail', region_name='us-east-1')
events = boto3.client('events', region_name='us-east-1')
sns = boto3.client('sns', region_name='us-east-1')
REGION = 'us-east-1'

account_id = boto3.client('sts').get_caller_identity()['Account']
print(f"Account: {account_id}")

In [ ]:
# Look up recent CloudTrail events — no trail needed, 90-day event history is free!
from datetime import datetime, timedelta, timezone

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(hours=1)

# Find all IAM events in the last hour
response = cloudtrail.lookup_events(
    LookupAttributes=[
        {'AttributeKey': 'EventSource', 'AttributeValue': 'iam.amazonaws.com'}
    ],
    StartTime=start_time,
    EndTime=end_time,
    MaxResults=10
)

print(f"Recent IAM API calls ({len(response['Events'])} found):")
for event in response['Events']:
    user = event.get('Username', 'Unknown')
    print(f"  {event['EventTime'].strftime('%H:%M:%S')}  {event['EventName']:40} by: {user}")

In [ ]:
# Create a CloudTrail trail (requires an S3 bucket)
import boto3

s3 = boto3.client('s3', region_name='us-east-1')
cloudtrail = boto3.client('cloudtrail', region_name='us-east-1')
account_id = boto3.client('sts').get_caller_identity()['Account']

TRAIL_BUCKET = f'cloudops-learning-trail-{account_id}'
TRAIL_NAME = 'cloudops-learning-trail'

# Create S3 bucket for trail logs
try:
    s3.create_bucket(Bucket=TRAIL_BUCKET)
    print(f"Created trail bucket: {TRAIL_BUCKET}")
except s3.exceptions.BucketAlreadyOwnedByYou:
    print(f"Trail bucket exists: {TRAIL_BUCKET}")

# Attach bucket policy (CloudTrail requires specific permissions)
bucket_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "AWSCloudTrailAclCheck",
            "Effect": "Allow",
            "Principal": {"Service": "cloudtrail.amazonaws.com"},
            "Action": "s3:GetBucketAcl",
            "Resource": f"arn:aws:s3:::{TRAIL_BUCKET}"
        },
        {
            "Sid": "AWSCloudTrailWrite",
            "Effect": "Allow",
            "Principal": {"Service": "cloudtrail.amazonaws.com"},
            "Action": "s3:PutObject",
            "Resource": f"arn:aws:s3:::{TRAIL_BUCKET}/AWSLogs/{account_id}/*",
            "Condition": {"StringEquals": {"s3:x-amz-acl": "bucket-owner-full-control"}}
        }
    ]
}

s3.put_bucket_policy(Bucket=TRAIL_BUCKET, Policy=json.dumps(bucket_policy))

# Create the trail
try:
    trail = cloudtrail.create_trail(
        Name=TRAIL_NAME,
        S3BucketName=TRAIL_BUCKET,
        IsMultiRegionTrail=True,       # Capture events from ALL regions
        IncludeGlobalServiceEvents=True  # IAM, STS, etc.
    )
    cloudtrail.start_logging(Name=TRAIL_NAME)  # Trails start paused!
    print(f"Trail created and logging started: {TRAIL_NAME}")
    print(f"Events will be delivered to: s3://{TRAIL_BUCKET}/")
except cloudtrail.exceptions.TrailAlreadyExistsException:
    print(f"Trail already exists: {TRAIL_NAME}")

In [ ]:
# Create EventBridge Rule: Alert when IAM user is created
# Pattern: CloudTrail event for CreateUser API call

# First, create an SNS topic for alerts
topic = sns.create_topic(Name='SecurityAlerts')
SECURITY_TOPIC_ARN = topic['TopicArn']
print(f"SNS Topic: {SECURITY_TOPIC_ARN}")

# Grant EventBridge permission to publish to SNS
sns.set_topic_attributes(
    TopicArn=SECURITY_TOPIC_ARN,
    AttributeName='Policy',
    AttributeValue=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "events.amazonaws.com"},
            "Action": "SNS:Publish",
            "Resource": SECURITY_TOPIC_ARN
        }]
    })
)

# Create EventBridge rule matching IAM CreateUser events
events.put_rule(
    Name='IAMUserCreated',
    EventPattern=json.dumps({
        "source": ["aws.iam"],
        "detail-type": ["AWS API Call via CloudTrail"],
        "detail": {
            "eventSource": ["iam.amazonaws.com"],
            "eventName": ["CreateUser"]
        }
    }),
    State='ENABLED',
    Description='Alert when a new IAM user is created'
)

# Add SNS as the target
events.put_targets(
    Rule='IAMUserCreated',
    Targets=[{
        'Id': 'SecurityAlertsSNS',
        'Arn': SECURITY_TOPIC_ARN
    }]
)

print("EventBridge rule 'IAMUserCreated' created.")
print("Any new IAM user creation will trigger an SNS notification.")

In [ ]:
# EventBridge Scheduler Rule — run something on a cron schedule
# Use case: nightly compliance report, daily backup verification

events.put_rule(
    Name='DailyComplianceCheck',
    ScheduleExpression='cron(0 6 * * ? *)',  # Every day at 6 AM UTC
    State='ENABLED',
    Description='Trigger daily compliance audit at 6 AM UTC'
)

print("Scheduled rule 'DailyComplianceCheck' created.")
print("Cron: 0 6 * * ? * = 6:00 AM UTC daily")
print()
print("Common cron patterns:")
print("  cron(0 12 * * ? *)    = Noon UTC daily")
print("  cron(0 8 ? * MON-FRI *)  = 8 AM weekdays")
print("  rate(5 minutes)       = Every 5 minutes (rate expression)")

## Exercise

In [ ]:
# TODO: Create an EventBridge rule that:
# - Fires when an EC2 instance TERMINATES (state = 'terminated')
# - Sends a notification to the 'SecurityAlerts' SNS topic
# - Rule name: 'EC2InstanceTerminated'
# Hint: Event source = 'aws.ec2', detail-type = 'EC2 Instance State-change Notification'

import boto3, json
events = boto3.client('events', region_name='us-east-1')
sns = boto3.client('sns', region_name='us-east-1')

SECURITY_TOPIC_ARN = sns.create_topic(Name='SecurityAlerts')['TopicArn']

# YOUR CODE HERE

In [ ]:
# Cleanup
import boto3

events_client = boto3.client('events', region_name='us-east-1')
sns_client = boto3.client('sns', region_name='us-east-1')
ct = boto3.client('cloudtrail', region_name='us-east-1')
s3_client = boto3.client('s3', region_name='us-east-1')

account_id = boto3.client('sts').get_caller_identity()['Account']
TRAIL_BUCKET = f'cloudops-learning-trail-{account_id}'

for rule_name in ['IAMUserCreated', 'DailyComplianceCheck', 'EC2InstanceTerminated']:
    try:
        targets = events_client.list_targets_by_rule(Rule=rule_name)['Targets']
        if targets:
            events_client.remove_targets(Rule=rule_name, Ids=[t['Id'] for t in targets])
        events_client.delete_rule(Name=rule_name)
        print(f"Deleted EventBridge rule: {rule_name}")
    except Exception as e:
        print(f"{rule_name}: {e}")

try:
    ct.stop_logging(Name='cloudops-learning-trail')
    ct.delete_trail(Name='cloudops-learning-trail')
    print("Deleted CloudTrail trail")
except Exception as e:
    print(f"Trail: {e}")

print(f"Note: Manually empty and delete S3 bucket '{TRAIL_BUCKET}' if needed (contains CloudTrail logs)")

## Summary

- **CloudTrail** records all AWS API calls — 90-day event history is free; trails extend to S3
- Multi-region trails are best practice — capture events from all regions
- **EventBridge** routes events using JSON pattern matching to targets (Lambda, SNS, SQS, etc.)
- CloudTrail + EventBridge = near-real-time security alerting
- EventBridge also supports **scheduled rules** (cron/rate expressions)

## Key Exam Points (SOA-C03)
- CloudTrail logs land in S3 within **~15 minutes** of the API call
- For **real-time** alerts from CloudTrail, use: CloudTrail → CloudWatch Logs → Metric Filter → Alarm → SNS OR CloudTrail → EventBridge
- **CloudTrail Insights** detects unusual write API activity
- `isMultiRegionTrail=True` is required for organization-wide coverage

---

**Next:** [Lesson 2.4 — Systems Manager Automation & Runbooks](lesson-2.4-systems-manager.ipynb)